## Code for Network Analysis 

In [1]:
from pathlib import Path
import sys

# --- Paths / imports -------------------------------------------------
PROJECT_ROOT = Path.cwd().parent
PREPROCESSING_DIR = PROJECT_ROOT / "functions" / "preprocessing"
for p in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(p) not in sys.path:
        sys.path.append(str(p))

import pickle
from server_config import redcap_path, preprocessed_path
import pandas as pd
import numpy as np
from scipy.stats import norm, rankdata

In [2]:

with open(preprocessed_path + '/ema_content.pkl', 'rb') as file:
    df_ema_content = pickle.load(file)  

with open(preprocessed_path + '/monitoring_data.pkl', 'rb') as file:
    df_monitoring = pickle.load(file)
    
df_redcap_final = pd.read_spss(redcap_path + "/sp1_cleaned" + "/baseline_raw_20251012.sav")
df_redcap_process = pd.read_csv(redcap_path + "/verlauf" + "/FOR_Verlauf.csv")
df_redcap_process_zert = pd.read_csv(redcap_path + "/verlauf" + "/FOR_Zertifizierung_Verlauf.csv")

In [3]:
# rename column
df_ema_content = df_ema_content.rename(
    columns={
        'n_beeps_beeps_completed_per_burst': 'n_beeps_completed_per_burst'
    }
)

In [4]:
# manual correction
replacements = {
    ('mASG', 1): 81,
    ('9UHX', 1): 70,
    ('LQFF', 1): 96,
    ('uxmL', 1): 114,
    ('mASG', 2): 80
}

for (pid, burst), value in replacements.items():
    
    mask = (
        (df_ema_content['id'] == pid) &
        (df_ema_content['measurement_burst'] == burst)
    )
    
    df_ema_content.loc[
        mask,
        'n_beeps_completed_per_burst'
    ] = value


In [5]:
# sanity check
#df_ema_content.loc[
#    (df_ema_content['measurement_burst'] == 2) &
#    (df_ema_content['id'] == 'mASG'),
#    ['timestamp_beep_completion']
#].nunique()


In [6]:
from server_config import datapath, proj_sheet, preprocessed_path, raw_path, backup_path


In [7]:
df_red_short = df_redcap_final[["for_id", "scid_cv_prim_cat", "age", "bdi_0_sum", "bsi_0_gsi", "bsi_0_sum"]]

In [8]:
col_list = ['for_id','t5_session_date','t1_session_date','t20_session_date','ic_date','t5_assessment_date',
 't20_assessment_date','post_assessment_date','dropout_date', 'ema_date']

In [9]:
df_redcap_process_short = df_redcap_process[col_list]
df_redcap_process_zert_short = df_redcap_process_zert[col_list]
df_redcap_process_short = pd.concat([df_redcap_process_short, df_redcap_process_zert_short], ignore_index=True)
df_redcap_process_short.dropna(subset= "ema_date", inplace=True)

In [10]:
df_monitoring = pd.read_csv(f"https://docs.google.com/spreadsheets/d/{proj_sheet}/export?format=csv")

In [11]:
df_monitoring = df_monitoring.copy()

df_monitoring.rename(columns = {"Pseudonym": "id",
                                "FOR_ID": "for_id",
                                "EMA_ID": "ema_id", 
                                "Status": "study_status",
                                "Studienversion":"study_version", 
                                "Start EMA Baseline": "ema_base_start", 
                                "Ende EMA Baseline": "ema_base_end", 
                                "Freischaltung/ Start EMA T20": "ema_t20_start",
                                "Ende EMA T20":"ema_t20_end", 
                                "Freischaltung/ Start EMA Post":"ema_post_start",
                               "Ende EMA Post":"ema_post_end", 
                                "T20=Post":"t20_post",
                               'Telefonat stattgefunden?': 'ema_phonecall',
                               "Status reduziert":"study_status_short",
                               'Dropout vor KV': "dropout_pre_14",
                                'Dropout nach KV':"dropout_post_14",
                                'Dropout Grund (frei)': "dropout_reason_text"
                               }, 
                     inplace=True)

df_monitoring = df_monitoring[['for_id', 'ema_id', 'id', 'study_version', 'study_status',
       't20_post', 'ema_base_start', 'ema_base_end', 'ema_t20_start', 'ema_t20_end',
       'ema_post_start', 'ema_post_end', "ema_phonecall","study_status_short", 'dropout_pre_14', 'dropout_post_14',
       'dropout_reason_text',  "ema_phonecall", "t20_post"]]

df_monitoring["id"] = df_monitoring["id"].str[:4]
df_monitoring["for_id"] = df_monitoring.for_id.str.strip()

df_monitoring["ema_base_start"] = pd.to_datetime(
    df_monitoring["ema_base_start"], dayfirst=True, errors="coerce", utc=True
)
df_monitoring["ema_base_end"] = pd.to_datetime(
    df_monitoring["ema_base_end"], dayfirst=True, errors="coerce", utc=True
)


#### Keep Columns necessary for Network Analysis

In [12]:
df_network = df_ema_content[['id', 'for_id', 'measurement_burst','item','response','beep_per_person_id', 'n_beeps_completed_per_burst']]

### Pivot dataframe 

In [13]:
# Keep PANAS items and all items starting with er_
df_ema_items = df_network[
    df_network["item"].str.startswith(("panas_", "er_"), na=False)
].copy()

extra_cols = ["measurement_burst", "for_id",  "n_beeps_completed_per_burst"]

# Keep one row per id + beep_per_person_id with the extra columns
aggregated_info = (
    df_ema_items
    .groupby(["id", "beep_per_person_id"])[extra_cols]
    .first()
    .reset_index()
)

# Pivot item responses to wide format
df_items_piv = df_ema_items.pivot_table(
    index=["id", "beep_per_person_id"],
    columns="item",
    values="response",
    aggfunc="first"
)

# Flatten columns
df_items_piv.columns = [col for col in df_items_piv.columns]

# Reset index
df_items_piv = df_items_piv.reset_index()

# Merge extra columns back in
df_items_piv = df_items_piv.merge(
    aggregated_info,
    on=["id", "beep_per_person_id"],
    how="left"
)

# Optional: remove duplicates
df_items_piv = df_items_piv.drop_duplicates()

In [14]:
na_cols = ['panas_fear1', 'panas_fear2', 'panas_guilt1','panas_guilt2', 'panas_hostility1', 'panas_hostility2','panas_sadness1', 'panas_sadness2','panas_loneliness']
pa_cols = ['panas_attentiveness', 'panas_joviality1', 'panas_joviality2','panas_selfassurance','panas_serenity1', 'panas_serenity2']

df_items_piv[pa_cols + na_cols] = df_items_piv[pa_cols + na_cols].apply(
    pd.to_numeric, errors="coerce"
)

In [15]:
group_cols = ['id', 'beep_per_person_id', 'measurement_burst']

scale_defs = {
    'panas_na':        na_cols,
    'panas_pa':        pa_cols 
}
for new_col, cols in scale_defs.items():
    df_items_piv[new_col] = (
        df_items_piv
        .groupby(group_cols)[cols]
        .transform('mean')
        .mean(axis=1)
    )

In [16]:
df_items_piv.n_beeps_completed_per_burst.describe()

count    47307.000000
mean        79.040353
std         25.304750
min          0.000000
25%         63.000000
50%         82.000000
75%        100.000000
max        127.000000
Name: n_beeps_completed_per_burst, dtype: float64

In [17]:
df_items_piv[df_items_piv.n_beeps_completed_per_burst > 130].id.unique()

array([], dtype=object)

In [18]:
df_mon_short = df_monitoring[["for_id", "study_version", "study_status","study_status_short",'dropout_pre_14', 'dropout_post_14',
       'dropout_reason_text', "ema_phonecall", "t20_post"]]
df_final = pd.merge(df_mon_short, df_red_short, on="for_id")
df_ema_short = df_ema_content[["for_id", "measurement_burst", "n_beeps_completed_per_burst"]].drop_duplicates(subset=["for_id", "measurement_burst"])
df_final = df_final.merge(df_ema_short, on="for_id")
df_final.to_csv(preprocessed_path + "/daten_pastushenko_corrected.csv")

In [19]:
#df_items_piv.to_csv("data_doro_network.csv")
df_items_piv.to_csv(
    preprocessed_path + "/data_doro_network.csv",
    index=False
)

In [25]:
df_final_till = df_final[df_final.measurement_burst == 0]

In [28]:
df_final_till = df_final_till[["for_id","n_beeps_completed_per_burst"]].drop_duplicates(subset=["for_id"])

In [31]:
df_final_till = df_final_till[df_final_till.n_beeps_completed_per_burst > 0]

In [ ]:
#df_final_till.to_csv("df_final_till.csv", index=False)

### Create artifical data for sharing

In [20]:
first_cols = ["id","for_id", "measurement_burst","beep_per_person_id",  "n_beeps_beeps_completed_per_burst"]
panas_cols = sorted([c for c in df_items_piv.columns if c.startswith("panas_")])
er_cols = sorted([c for c in df_items_piv.columns if c.startswith("er_")])
other_cols = sorted([c for c in df_items_piv.columns if c not in first_cols + panas_cols + er_cols])

df_items_piv = df_items_piv[first_cols + panas_cols + er_cols + other_cols]
df_items_piv.drop(['panas_fatigue', 'panas_fear1', 'panas_fear2', 'panas_guilt1',
       'panas_guilt2', 'panas_hostility1', 'panas_hostility2',
       'panas_joviality1', 'panas_joviality2', 'panas_loneliness',
       'panas_sadness1', 'panas_sadness2', 'panas_selfassurance',
       'panas_serenity1', 'panas_serenity2', 'panas_shyness','panas_attentiveness'], axis=1, inplace=True)

KeyError: "['n_beeps_beeps_completed_per_burst'] not in index"

In [ ]:
import numpy as np
import pandas as pd
import string
from scipy.stats import norm, rankdata

# =========================================================
# 1) SETTINGS
# =========================================================
SEED = 42
rng = np.random.default_rng(SEED)

# choose a range for date shifting when anonymizing beep_per_person_id
# example: between -30 and +30 days
MIN_DAY_SHIFT = -30
MAX_DAY_SHIFT = 30

# =========================================================
# 2) PREP ORIGINAL DATA
# =========================================================
df_tmp = df_items_piv.copy()

# parse beep_per_person_id into date part and beep suffix
# e.g. "20231010_2.0" -> beep_date = 2023-10-10, beep_suffix = "2.0"
if "beep_per_person_id" in df_tmp.columns:
    parts = df_tmp["beep_per_person_id"].astype(str).str.split("_", n=1, expand=True)
    df_tmp["beep_date_str"] = parts[0]
    df_tmp["beep_suffix"] = parts[1]
    df_tmp["beep_date"] = pd.to_datetime(df_tmp["beep_date_str"], format="%Y%m%d", errors="coerce")

# =========================================================
# 3) CHOOSE COLUMNS TO SIMULATE
# =========================================================
item_cols = [c for c in df_tmp.columns if c.startswith("panas_") or c.startswith("er_")]

extra_sim_cols = [
    c for c in ["n_beeps_beeps_completed_per_burst"]
    if c in df_tmp.columns
]

sim_cols = item_cols + extra_sim_cols

structural_cols = [
    c for c in ["id", "for_id", "beep_per_person_id", "measurement_burst", "date", "beep_date", "beep_suffix"]
    if c in df_tmp.columns
]

# keep only rows with complete simulated variables
df_complete = df_tmp[structural_cols + sim_cols].dropna(subset=sim_cols).reset_index(drop=True)

# data used for fitting the simulator
df_sim_input = df_complete[sim_cols].copy()

print("N rows used for simulation:", len(df_sim_input))
print("Columns simulated:", sim_cols)
print("Structural columns retained from original data:", structural_cols)

# =========================================================
# 4) HELPERS
# =========================================================
def make_psd(corr, eps=1e-6):
    """Force correlation matrix to be positive semidefinite."""
    vals, vecs = np.linalg.eigh(corr)
    vals = np.clip(vals, eps, None)
    corr_psd = vecs @ np.diag(vals) @ vecs.T
    d = np.sqrt(np.diag(corr_psd))
    corr_psd = corr_psd / np.outer(d, d)
    np.fill_diagonal(corr_psd, 1.0)
    return corr_psd


def fit_gaussian_copula(df, categorical_cols=None):
    """
    Fit a Gaussian copula model.
    - numeric columns: rank-based normal transform
    - categorical columns: probabilities mapped to latent normal cutpoints
    """
    if categorical_cols is None:
        categorical_cols = []

    df = df.copy().reset_index(drop=True)
    cols = df.columns.tolist()
    meta = {}
    Z = np.zeros((len(df), len(cols)), dtype=float)

    for j, col in enumerate(cols):
        x = df[col]

        is_categorical = (
            col in categorical_cols
            or pd.api.types.is_object_dtype(x)
            or isinstance(x.dtype, pd.CategoricalDtype)
        )

        if is_categorical:
            categories = pd.Series(x).dropna().unique().tolist()
            probs = pd.Series(x).value_counts(normalize=True).reindex(categories).values
            cum_probs = np.cumsum(probs)

            lower = np.r_[0, cum_probs[:-1]]
            upper = cum_probs
            mids = (lower + upper) / 2
            mids = np.clip(mids, 1e-6, 1 - 1e-6)

            latent_map = {cat: norm.ppf(mid) for cat, mid in zip(categories, mids)}
            Z[:, j] = pd.Series(x).map(latent_map).astype(float).values

            meta[col] = {
                "type": "categorical",
                "categories": categories,
                "cum_probs": cum_probs,
            }

        else:
            x_num = pd.to_numeric(x, errors="coerce").astype(float).values
            ranks = rankdata(x_num, method="average")
            u = (ranks - 0.5) / len(x_num)
            u = np.clip(u, 1e-6, 1 - 1e-6)
            z = norm.ppf(u)

            Z[:, j] = z
            meta[col] = {
                "type": "continuous",
                "sorted_values": np.sort(x_num),
                "is_integer": np.all(np.isclose(x_num, np.round(x_num))),
                "min": np.nanmin(x_num),
                "max": np.nanmax(x_num),
            }

    corr = np.corrcoef(Z, rowvar=False)
    corr = make_psd(corr)

    return {"columns": cols, "meta": meta, "corr": corr}


def sample_gaussian_copula(fitted_model, n_samples, seed=42):
    """Generate simulated data from the fitted Gaussian copula model."""
    rng_local = np.random.default_rng(seed)
    cols = fitted_model["columns"]
    meta = fitted_model["meta"]
    corr = fitted_model["corr"]

    latent = rng_local.multivariate_normal(
        mean=np.zeros(len(cols)),
        cov=corr,
        size=n_samples
    )

    sim = pd.DataFrame(index=np.arange(n_samples))

    for j, col in enumerate(cols):
        info = meta[col]
        u = norm.cdf(latent[:, j])

        if info["type"] == "continuous":
            vals = info["sorted_values"]
            sim_col = np.quantile(vals, u, method="linear")

            if info["is_integer"]:
                sim_col = np.rint(sim_col)
                sim_col = np.clip(sim_col, info["min"], info["max"]).astype(int)

            sim[col] = sim_col

        else:
            idx = np.searchsorted(info["cum_probs"], u, side="right")
            idx = np.clip(idx, 0, len(info["categories"]) - 1)
            sim[col] = np.array(info["categories"], dtype=object)[idx]

    return sim


def generate_unique_ids(n, length=4, seed=42):
    """
    Generate unique IDs like '05kz' using a-z, A-Z, 0-9.
    """
    rng_local = np.random.default_rng(seed)
    alphabet = np.array(list(string.ascii_letters + string.digits))

    ids = set()
    while len(ids) < n:
        new_id = "".join(rng_local.choice(alphabet, size=length))
        ids.add(new_id)

    return list(ids)


def generate_unique_for_ids(n, seed=42):
    """
    Generate unique FOR IDs like FOR13015.
    """
    rng_local = np.random.default_rng(seed)
    ids = set()

    while len(ids) < n:
        num = rng_local.integers(10000, 100000)  # 5 digits
        ids.add(f"FOR{num}")

    return list(ids)


# =========================================================
# 5) FIT + SIMULATE SUBSTANTIVE VARIABLES
# =========================================================
categorical_cols = []

fitted = fit_gaussian_copula(
    df_sim_input,
    categorical_cols=categorical_cols
)

df_item_piv_sim = sample_gaussian_copula(
    fitted_model=fitted,
    n_samples=len(df_sim_input),
    seed=SEED
)

# =========================================================
# 6) KEEP ORIGINAL STRUCTURAL ROWS
# =========================================================
df_struct = df_complete[structural_cols].reset_index(drop=True).copy()

# =========================================================
# 7) SIMULATE PERSON-LEVEL id AND for_id
# =========================================================
# one new id and one new for_id per original participant
if "id" in df_struct.columns:
    unique_orig_ids = pd.Series(df_struct["id"].dropna().unique()).tolist()

    new_ids = generate_unique_ids(len(unique_orig_ids), length=4, seed=SEED)
    new_for_ids = generate_unique_for_ids(len(unique_orig_ids), seed=SEED + 1)

    id_map = dict(zip(unique_orig_ids, new_ids))
    for_id_map = dict(zip(unique_orig_ids, new_for_ids))

    df_struct["id"] = df_struct["id"].map(id_map)

    if "for_id" in df_struct.columns:
        df_struct["for_id"] = df_complete["id"].map(for_id_map).values

# =========================================================
# 8) ANONYMIZE beep_per_person_id
# =========================================================
# Shift dates within each original person x measurement_burst group
# while preserving the within-group pattern and original beep suffix.

if {"beep_date", "beep_suffix", "measurement_burst"}.issubset(df_complete.columns):
    # use original participant IDs for grouping, before replacement
    group_keys = ["id", "measurement_burst"]
    df_group_source = df_complete[group_keys].copy()

    unique_groups = df_group_source.drop_duplicates().reset_index(drop=True)
    unique_groups["day_shift"] = rng.integers(
        MIN_DAY_SHIFT,
        MAX_DAY_SHIFT + 1,
        size=len(unique_groups)
    )

    df_group_source = df_group_source.merge(unique_groups, on=group_keys, how="left")

    shifted_dates = (
        df_complete["beep_date"].reset_index(drop=True) +
        pd.to_timedelta(df_group_source["day_shift"], unit="D")
    )

    shifted_date_str = shifted_dates.dt.strftime("%Y%m%d")
    beep_suffix = df_complete["beep_suffix"].reset_index(drop=True).astype(str)

    df_struct["beep_per_person_id"] = shifted_date_str + "_" + beep_suffix

    # also anonymize date column consistently if it exists
    if "date" in df_struct.columns:
        orig_date = pd.to_datetime(df_complete["date"], errors="coerce")
        df_struct["date"] = orig_date + pd.to_timedelta(df_group_source["day_shift"], unit="D")

# =========================================================
# 9) COMBINE STRUCTURE + SIMULATED VARIABLES
# =========================================================
df_item_piv_sim = pd.concat(
    [df_struct.reset_index(drop=True), df_item_piv_sim.reset_index(drop=True)],
    axis=1
)

# remove helper columns if present
drop_helper_cols = [c for c in ["beep_date", "beep_suffix"] if c in df_item_piv_sim.columns]
if drop_helper_cols:
    df_item_piv_sim = df_item_piv_sim.drop(columns=drop_helper_cols)

# =========================================================
# 10) OPTIONAL: REORDER COLUMNS
# =========================================================
first_cols = [
    c for c in [
        "id",
        "for_id",
        "beep_per_person_id",
        "measurement_burst",
        "date",
        "n_beeps_beeps_completed_per_burst"
    ]
    if c in df_item_piv_sim.columns
]

other_cols = [c for c in df_item_piv_sim.columns if c not in first_cols]
df_item_piv_sim = df_item_piv_sim[first_cols + other_cols]


In [ ]:
df_item_piv_sim.to_csv(preprocessed_path + "/simulated_ema_network.csv", index=False)